# ДЗ 7. Срезы orders.csv: groupby, pivot_table, merge
Срок: **до воскресенья 29.11, 24:00**.
Сдача: папка `hw07` репозитория `dsa-homework` (`hw07.ipynb`): `git add hw07` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать. Рядом с каждой долей или средним указывайте, на скольких заказах оно посчитано.

Инструменты: всё с уроков 1–7. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))
```

Ваши ответы и объяснения:

1. 4,2 по с группируем и находим сумму всех группированных элементов
2. 3,2 - count() None включает и считает все; а size() только уникальные (после поняла, что забыла группировку)
3. было тяжело представить, пришлось посмотреть на результат 
4. 2 3 - a.merge(b, on="id") по умолчанию стоит иннер, который неповторяющиеся строки не показывает, поэтому сокращается до 2. how="left" позволит оставить все строки левой таблицы, то есть а 
5. думала 3, но на самом деле 4 - для левой таблицы продублируется 1, 1 из второй таблицы

In [21]:
import pandas as pd


In [23]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("/Users/gulsatkalbayeva/Documents/dsa-homework/orders.csv"), "| customers.csv:", os.path.exists("/Users/gulsatkalbayeva/Documents/dsa-homework/customers.csv"))

Положите orders.csv и customers.csv в одну папку с ноутбуком
orders.csv: True | customers.csv: True


In [25]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("/Users/gulsatkalbayeva/Documents/dsa-homework/orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [27]:
df["cancel"] = df["status"] == "отменён"
paid = df[df["status"] == "оплачен"]
cu = pd.read_csv("/Users/gulsatkalbayeva/Documents/dsa-homework/customers.csv")
len(paid), cu.shape

(4739, (2200, 4))

Можно вместо ячейки очистки взять свою функцию `clean()` из ДЗ 6. Проверка: 5 280 строк, 4 739 оплаченных.

## [База] 1. Сводка по городам одной таблицей
Через `agg` с именами: число заказов, доля отмен, средний срок доставки по городам. Добавьте столбец «выручка оплаченных»: `by_city["revenue"] = paid.groupby("city")["amount"].sum()` (Pandas выровняет по ярлыкам, урок 5). Отсортируйте по выручке.
Какой город выделяется сразу по двум показателям?

In [ ]:
# 1 - выделяются Шымкент - каждый 5 заказ отмена, долгая доставка и также Актобе - хоть и доставка быстрая, 
# как в других городах, отмена каждый десятый заказ
by_city = df.groupby('city').agg(count=('amount', 'count'), cancel_rate = ('cancel', 'mean'), average_delivery = ('delivery_days', 'mean'))
by_city["revenue"] = paid.groupby("city")["amount"].sum()
by_city.sort_values('revenue', ascending=False).round(2)

,count,cancel_rate,average_delivery,revenue
city,,,,
Алматы,2116,0.08,2.00,29144090.0
Астана,1339,0.09,2.02,17884750.0
Шымкент,776,0.21,4.45,8602660.0
Караганда,519,0.09,1.94,7020200.0
Актобе,387,0.11,2.07,5117380.0
Атырау,143,0.08,2.04,1893850.0


## [База] 2. ДЗ 4 одной строкой
В ДЗ 4 вы считали категории циклами и NumPy. Повторите через `pivot_table`: выручка оплаченных по категориям и месяцам, изменение в %, средний чек по категориям и месяцам.
Сверьте с ответами ДЗ 4: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре 33 600 ₸. Сколько строк кода было и сколько стало?

In [64]:
# 2 ALL
p= paid.pivot_table(values='amount', index='category', columns='month', aggfunc=['sum','mean'])
p["change_%"]=(p["sum"][10]/p["sum"][9]-1)*100
p.sort_values('change_%').round(2)

/var/folders/w6/l182g06s4699q3wn35dlj0240000gn/T/ipykernel_50787/3362458079.py:3: Pandas4Warning: Setting a new column on a DataFrame with a MultiIndex using a key that is not a full-length tuple is deprecated and will raise in a future version. Use a full-length tuple key instead.
  p["change_%"]=(p["sum"][10]/p["sum"][9]-1)*100


sum                  mean           change_%
month                 9          10         9        10         
category                                                        
Одежда        9167100.0   8505730.0  13520.80  13045.60    -7.21
Электроника  17402330.0  16968100.0  32772.75  33600.20    -2.50
Дом           4997730.0   5125790.0  10499.43  10768.47     2.56
Красота       2610950.0   2678990.0   6187.09   6158.60     2.61
Книги         1065300.0   1140910.0   4020.00   3815.75     7.10

## [База] 3. Отмены и срок доставки
По `delivery_days`: число заказов и доля отмен. Во сколько раз отменяют чаще при доставке за 8 дней, чем за 1 день?
Что это значит для Шымкента, где в октябре доставка 5–8 дней?

In [73]:
# 3 
delivery = df.groupby('delivery_days').agg(count = ('amount', 'count'), cancel_rate = ('cancel', 'mean'))
delivery

,count,cancel_rate
delivery_days,,
1.0,1557,0.042389
2.0,1570,0.079618
3.0,1590,0.133962
5.0,86,0.220930
6.0,120,0.216667
7.0,95,0.347368
8.0,104,0.432692


In [77]:
print("За 8 дней доставку отменяют в", round(delivery.loc[8.0, 'cancel_rate']/delivery.loc[1.0, 'cancel_rate'],2), "раз чаще")

За 8 дней доставку отменяют в 10.21 раз чаще


## [Стандарт] 4. Исправьте код от AI
AI посчитал изменение выручки по городам и пишет менеджеру: «В Шымкенте выручка выросла на 2,4%, проблем нет». В коде **две** ошибки. Найдите их, исправьте и напишите менеджеру правильную цифру.
```python
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))
```

In [88]:
# 4
p = paid.pivot_table(index="city", columns="month", values="amount", aggfunc='sum') #здесь по умолчанию показывает mean если не указать aggfunc
#также мы считали по таблице df с неоплаченными заказами: фактически считали несуществующие деньги
p["chg"] = (p[10] / p[9] - 1)*100
print(p["chg"].round(3))

city
Актобе       -1.418
Алматы        0.979
Астана       -2.033
Атырау        4.964
Караганда    -7.588
Шымкент     -11.289
Name: chg, dtype: float64


## [Стандарт] 5. Возраст клиентов
Соедините заказы с клиентами (`merge`, `how="left"`, проверьте `len`). Возрастная группа: `m["age_group"] = m["age"] // 10 * 10`.
По группам: число заказов, доля отмен, средний и медианный чек оплаченных.
AI предлагает: «Группа 60 покупает дороже всех и чаще отменяет, сделаем для неё отдельную акцию». Какие возрасты попали в группу 60 и сколько в ней заказов? Подтверждает ли медиана вывод AI?

In [102]:
# 5
m=paid.merge(cu, on='customer_id', how='left')
len(m), len(cu), len(paid)
m['customer_id'].isna().sum()
m["age_group"] = m["age"] // 10 * 10
p1 = m.groupby('age_group').agg(count = ('amount', 'count'),mean=('amount', 'mean'),median=('amount', 'median'))
m2=df.merge(cu, on='customer_id', how='left')
m2["age_group"] = m["age"] // 10 * 10
p1['cancel_rate'] = m2.groupby('age_group').agg(cancel_rate=('cancel', 'mean'))
p1

,count,mean,median,cancel_rate
age_group,,,,
10,209,14044.736842,10700.0,0.081340
20,1108,14130.415162,10100.0,0.088448
30,1103,15142.339075,10670.0,0.099728
40,1040,14792.971154,10050.0,0.098077
50,1175,14671.795745,10240.0,0.115745
60,104,16779.134615,10495.0,0.125000


In [108]:
print("Медиана показывает, что цена среднего чека не сильно отличается при возрастах. \nПоэтому утверждать, что вся категория покупает дорого нельзя; \nтакой высокий средний чек объясняется выбросами: должны быть малое количество клиентов,\nкоторые закупают на большую сумму. В эту категорию попали люди от 61 до 69 лет")

Медиана показывает, что цена среднего чека не сильно отличается при возрастах. 
Поэтому утверждать, что вся категория покупает дорого нельзя; 
такой высокий средний чек объясняется выбросами: должны быть малое количество клиентов,
которые закупают на большую сумму. В эту категорию попали люди от 61 до 69 лет


## [Стандарт] 6. Повторные покупки
Сколько заказов у каждого клиента (`groupby("customer_id").size()`)? Сколько клиентов сделали 1, 2, 3… заказа (`value_counts().sort_index()`)? Какая доля клиентов купила больше одного раза?
Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? Сколько клиентов из customers.csv не заказали ничего?

In [134]:
# 6
ord=m.groupby('customer_id').size().value_counts().sort_index()
one=ord.iloc[0]
all=ord.sum()
one_share = one/all*100
rest = round(100 - one_share,2)
print(f"{rest}% клиентов купила больше одного раза")

70.59% клиентов купила больше одного раза


In [138]:
#Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? 
# Сколько клиентов из customers.csv не заказали ничего?
n = m.groupby("customer_id")["month"].nunique()
print(f"{(n==2).sum()} клиентов заказывали и в сентябре, и в октябре")

946 клиентов заказывали и в сентябре, и в октябре


In [141]:
no_orders = ~cu["customer_id"].isin(df["customer_id"])
print(no_orders.sum(), "не сделали заказ")

198 не сделали заказ


## [Челлендж] 7. Сколько стоил сбой доставки в Шымкенте
Допустим, в октябре в Шымкенте было бы столько же заказов и тот же средний чек оплаченных, а доля отмен осталась как в сентябре. Какой была бы выручка Шымкента в октябре? Сколько компания потеряла из-за отмен?
Сравните потерю с падением всей компании (823 890 ₸). Что было бы с выручкой компании без сбоя?

In [ ]:
# 7

## [Челлендж] 8. Органика −6,0%: сигнал или шум?
Разбейте клиентов из customers.csv **случайно** на три группы в тех же пропорциях, что и каналы (`np.random.default_rng(7)`, `rng.choice`, `map`). Посчитайте изменение выручки оплаченных окт/сент для каждой группы. Повторите 100 раз в цикле.
В каком диапазоне меняется выручка случайной группы размером с органику (5% и 95% перцентили)? Можно ли по «органика −6,0%» делать вывод о SEO?

In [ ]:
# 8